# Day 3: Data Aggregation and Grouping

Today we'll learn about one of pandas' most powerful features: grouping and aggregation operations.

In [ ]:
import pandas as pd
import numpy as np

# Create comprehensive sample data
np.random.seed(42)
data = {
    'Employee': ['Alice', 'Bob', 'Charlie', 'David', 'Eva', 'Frank', 'Grace', 'Henry'],
    'Department': ['IT', 'Finance', 'IT', 'HR', 'Finance', 'IT', 'HR', 'Finance'],
    'City': ['New York', 'London', 'New York', 'Paris', 'London', 'Tokyo', 'Paris', 'Tokyo'],
    'Age': [25, 30, 35, 28, 32, 45, 29, 38],
    'Salary': [50000, 60000, 70000, 55000, 65000, 80000, 58000, 72000],
    'Experience': [2, 5, 8, 3, 6, 15, 4, 10],
    'Performance': ['Good', 'Excellent', 'Good', 'Average', 'Excellent', 'Good', 'Average', 'Excellent']
}
df = pd.DataFrame(data)
print("Sample DataFrame:")
print(df)

## 1. Basic Grouping

In [ ]:
# Group by single column
dept_groups = df.groupby('Department')
print("Groups by Department:")
for name, group in dept_groups:
    print(f"\n{name}:")
    print(group)
print()

# Group sizes
print("Group sizes:")
print(df.groupby('Department').size())
print()

# Get specific group
print("IT Department only:")
print(dept_groups.get_group('IT'))

## 2. Aggregation Functions

In [ ]:
# Basic aggregations
print("Mean salary by department:")
print(df.groupby('Department')['Salary'].mean())
print()

print("Sum of salaries by department:")
print(df.groupby('Department')['Salary'].sum())
print()

print("Count by department:")
print(df.groupby('Department')['Salary'].count())
print()

# Multiple aggregations
print("Multiple statistics:")
print(df.groupby('Department')['Salary'].agg(['mean', 'median', 'std', 'min', 'max']))
print()

# Aggregation on multiple columns
print("Multiple columns aggregation:")
print(df.groupby('Department')[['Salary', 'Age', 'Experience']].mean())

## 3. Multiple Group By

In [ ]:
# Group by multiple columns
print("Group by Department and City:")
multi_group = df.groupby(['Department', 'City'])['Salary'].mean()
print(multi_group)
print()

# Unstack for better visualization
print("Unstacked view:")
print(multi_group.unstack())
print()

# Fill NaN values
print("Unstacked with filled NaN:")
print(multi_group.unstack(fill_value=0))

## 4. Custom Aggregation Functions

In [ ]:
# Custom function
def salary_range(series):
    return series.max() - series.min()

print("Salary range by department:")
print(df.groupby('Department')['Salary'].agg(salary_range))
print()

# Multiple custom functions
def coefficient_of_variation(series):
    return series.std() / series.mean()

print("Custom aggregations:")
custom_aggs = df.groupby('Department')['Salary'].agg([
    ('Range', salary_range),
    ('CV', coefficient_of_variation),
    ('Mean', 'mean')
])
print(custom_aggs)
print()

# Lambda functions
print("Using lambda functions:")
print(df.groupby('Department')['Salary'].agg([
    ('Mean', 'mean'),
    ('Above_60k', lambda x: (x > 60000).sum()),
    ('Median_Deviation', lambda x: abs(x - x.median()).mean())
]))

## 5. Different Aggregations for Different Columns

In [ ]:
# Different aggregations for different columns
agg_dict = {
    'Salary': ['mean', 'median', 'std'],
    'Age': ['min', 'max', 'mean'],
    'Experience': ['sum', 'mean'],
    'Employee': 'count'
}

print("Different aggregations for different columns:")
result = df.groupby('Department').agg(agg_dict)
print(result)
print()

# Flatten column names
result.columns = ['_'.join(col).strip() for col in result.columns.values]
print("With flattened column names:")
print(result)

## 6. Transform and Apply

In [ ]:
# Transform - returns same shape as original
df['Salary_Normalized'] = df.groupby('Department')['Salary'].transform(
    lambda x: (x - x.mean()) / x.std()
)

df['Salary_Rank'] = df.groupby('Department')['Salary'].rank(ascending=False)

print("With transformed columns:")
print(df[['Employee', 'Department', 'Salary', 'Salary_Normalized', 'Salary_Rank']])
print()

# Apply - can return different shapes
def dept_summary(group):
    return pd.Series({
        'count': len(group),
        'avg_salary': group['Salary'].mean(),
        'top_performer': group.loc[group['Salary'].idxmax(), 'Employee']
    })

print("Department summary using apply:")
print(df.groupby('Department').apply(dept_summary))

## 7. Filtering Groups

In [ ]:
# Filter groups based on group properties
print("Departments with more than 2 employees:")
large_depts = df.groupby('Department').filter(lambda x: len(x) > 2)
print(large_depts)
print()

print("Departments with average salary > 60000:")
high_salary_depts = df.groupby('Department').filter(lambda x: x['Salary'].mean() > 60000)
print(high_salary_depts[['Employee', 'Department', 'Salary']])

## 8. Pivot Tables

In [ ]:
# Basic pivot table
print("Basic pivot table:")
pivot1 = df.pivot_table(values='Salary', index='Department', columns='Performance', aggfunc='mean')
print(pivot1)
print()

# Multiple values
print("Pivot table with multiple values:")
pivot2 = df.pivot_table(values=['Salary', 'Age'], index='Department', columns='Performance', aggfunc='mean')
print(pivot2)
print()

# Multiple aggregation functions
print("Multiple aggregation functions:")
pivot3 = df.pivot_table(values='Salary', index='Department', columns='Performance', 
                       aggfunc=['mean', 'count'], fill_value=0)
print(pivot3)
print()

# With margins (totals)
print("With margins:")
pivot4 = df.pivot_table(values='Salary', index='Department', columns='Performance', 
                       aggfunc='mean', margins=True, fill_value=0)
print(pivot4)

## 9. Cross-tabulation

In [ ]:
# Basic cross-tabulation
print("Cross-tabulation of Department and Performance:")
crosstab1 = pd.crosstab(df['Department'], df['Performance'])
print(crosstab1)
print()

# With percentages
print("Cross-tabulation with percentages:")
crosstab2 = pd.crosstab(df['Department'], df['Performance'], normalize='index')
print(crosstab2.round(3))
print()

# With values
print("Cross-tabulation with values (average salary):")
crosstab3 = pd.crosstab(df['Department'], df['Performance'], values=df['Salary'], aggfunc='mean')
print(crosstab3.round(0))
print()

# With margins
print("Cross-tabulation with margins:")
crosstab4 = pd.crosstab(df['Department'], df['Performance'], margins=True)
print(crosstab4)

## 10. Advanced Grouping Techniques

In [ ]:
# Grouping by custom criteria
def age_group(age):
    if age < 30:
        return 'Young'
    elif age < 40:
        return 'Middle'
    else:
        return 'Senior'

print("Grouping by custom age groups:")
age_salary = df.groupby(df['Age'].apply(age_group))['Salary'].mean()
print(age_salary)
print()

# Grouping by multiple custom criteria
print("Grouping by age group and performance:")
multi_custom = df.groupby([df['Age'].apply(age_group), 'Performance'])['Salary'].mean()
print(multi_custom)
print()

# Using cut for binning
df['Salary_Bin'] = pd.cut(df['Salary'], bins=3, labels=['Low', 'Medium', 'High'])
print("Salary distribution by department:")
salary_dist = pd.crosstab(df['Department'], df['Salary_Bin'])
print(salary_dist)

## Practice Exercises

In [ ]:
# Exercise 1: Find the department with highest average experience
avg_exp_by_dept = df.groupby('Department')['Experience'].mean()
highest_exp_dept = avg_exp_by_dept.idxmax()
print(f"Department with highest average experience: {highest_exp_dept}")
print(f"Average experience: {avg_exp_by_dept[highest_exp_dept]:.2f} years")
print()

# Exercise 2: Create a summary report
summary_report = df.groupby('Department').agg({
    'Employee': 'count',
    'Salary': ['mean', 'min', 'max'],
    'Age': 'mean',
    'Experience': 'mean'
})
summary_report.columns = ['Employee_Count', 'Avg_Salary', 'Min_Salary', 'Max_Salary', 'Avg_Age', 'Avg_Experience']
print("Department Summary Report:")
print(summary_report.round(2))
print()

# Exercise 3: Find top performer in each department
top_performers = df.loc[df.groupby('Department')['Salary'].idxmax()]
print("Top performers by department:")
print(top_performers[['Department', 'Employee', 'Salary']])

## Tomorrow's Preview
In Day 4, we'll cover:
- Merging and Joining DataFrames
- Concatenation
- Reshaping Data (Melt, Stack, Unstack)
- Advanced Data Manipulation